### What Is Liveness Detection?

It means:

> Verify that the detected face belongs to a live human physically present.

There are 3 main categories:


#### 1. Passive Liveness (No User Action)


Examples:

- Texture analysis

- Depth cues

- Reflection detection

- Micro-expression patterns

Hard to implement properly. <br>
Requires additional models.

#### 2. Active Liveness (Challenge-Based)

System asks user to:

- Blink

- Turn head

- Smile

- Look left

This is much easier and realistic for us.

#### 3. Hardware-Based (Advanced)

- Infrared depth sensors

- 3D mapping

- Structured light (iPhone FaceID style)

We don’t have that.

#### We Will Implement:
Blink Detection (Active Liveness)

Because it's:

- Simple

- Effective against photos

- Practical

### Theory — How Blink Detection Works

We detect:

1. Eye landmarks

1. Compute Eye Aspect Ratio (EAR)

1. Detect rapid change    

### Eye Aspect Ratio (EAR)

If eye landmarks are:

<pre>
P1      P2
  \    /
   ----
  /    \
P4      P3
</pre>

We compute:

`EAR = (vertical distance) / (horizontal distance)`

When eye is:

- Open → EAR ≈ 0.25–0.35

- Closed → EAR ≈ 0.15 or lower

So if EAR drops below threshold briefly → blink detected.

### Important

MTCNN does not give detailed eye landmarks.

We need a facial landmark model.

Best simple option:

`mediapipe`

It gives:

- 468 face landmarks

- Real-time

- Lightweight

### Mediapipe Structure (Tasks API)

<pre>
Camera Frame
   ↓
Convert to Mediapipe Image
   ↓
Run FaceLandmarker
   ↓
Get 468 landmarks
   ↓
Extract eye points
   ↓
Compute EAR
   ↓
Decide blink or not
</pre>

### Install Model File (Required for Tasks API)



Mediapipe Tasks requires a model file.

Download:

👉 https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

Save it in your project folder.

Example:
<pre>
M5 Face Recognition/
    face_landmarker.task
</pre>

### Imports

In [6]:
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2

### Initialize FaceLandmarker

In [7]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

| Component               | What it Does                                                |
| ----------------------- | ----------------------------------------------------------- |
| `BaseOptions`           | Defines model-related configuration (like model file path). |
| `FaceLandmarker`        | The actual face landmark detection model.                   |
| `FaceLandmarkerOptions` | Configuration object for the landmarker.                    |
| `RunningMode`           | Specifies how the model runs (IMAGE / VIDEO / LIVE_STREAM). |


| Attribute            | Meaning                         |
| -------------------- | ------------------------------- |
| `model_asset_path`   | Path to trained model file      |
| `running_mode=VIDEO` | Optimized for sequential frames |
| `num_faces=1`        | Detect only 1 face              |


`landmarker = FaceLandmarker.create_from_options(options)`

Creates a face detection object using the configuration.

### EAR Calculation Function

In [8]:
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(eye_points):
    A = euclidean(eye_points[1], eye_points[5])
    B = euclidean(eye_points[2], eye_points[4])
    C = euclidean(eye_points[0], eye_points[3])

    ear = (A + B) / (2.0 * C)
    return ear

    def euclidean(p1, p2):
        return np.linalg.norm(np.array(p1) - np.array(p2))
        
What it does:

Calculates straight-line distance between 2 points.

Formula:

$\sqrt{(x_2−x_1)^2+(y_2−y_1)^2}$ <br> <br>
​
Used for:

- Measuring eye opening distance

def calculate_ear(eye_points):

EAR formula:

$\small{EAR}=\large{​\frac{A+B}{2C}}$ <br> <br>
	​
Where:

- A = vertical distance

- B = vertical distance

- C = horizontal distance

What it means:

| Eye State  | EAR Value         |
| ---------- | ----------------- |
| Open eye   | High (~0.25–0.35) |
| Closed eye | Low (~0.15–0.20)  |

So if EAR drops below threshold → blink.

### Blink Detection Variables

In [ ]:
EAR_THRESHOLD = 0.20
BLINK_FRAMES = 3

blink_counter = 0
blink_display_frames = 15   # show text for 20 frames
blink_display_counter = 0

HEAD_THRESHOLD = 15
head_movement_display_frames = 15
head_display_counter = 0
head_direction = ""

#### Blink Configuration Variables
| Variable        | Purpose                       |
| --------------- | ----------------------------- |
| `EAR_THRESHOLD` | Below this = eye closed       |
| `BLINK_FRAMES`  | Must stay closed for 3 frames |

#### Blink Counters
| Variable                | Purpose                             |
| ----------------------- | ----------------------------------- |
| `blink_counter`         | Counts consecutive closed frames    |
| `blink_display_frames`  | Show "Blink Detected" for 15 frames |
| `blink_display_counter` | Controls display duration           |

#### Head Movement Variables
| Variable               | Meaning                   |
| ---------------------- | ------------------------- |
| `HEAD_THRESHOLD`       | Minimum pixel offset from center to consider head turned.    |
| `head_direction`       | Text to display (Left, Right, Center)           |
| `head_display_counter` | Timer for showing head movement text |
| `head_movement_display_frames`    | Display duration  |


### Camera Loop

Mediapipe needs its own Image format.

In [10]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame_rgb
    )

    timestamp = int(time.time() * 1000)

    result = landmarker.detect_for_video(mp_image, timestamp)
    
    # extract eye and nose landmarks
    if result.face_landmarks:

        h, w, _ = frame.shape
        landmarks = result.face_landmarks[0]
        
        # nose landmarks
        nose = landmarks[1]  # nose tip
        nose_x = int(nose.x * w)

        frame_center_x = w // 2

        offset = nose_x - frame_center_x

        if offset > HEAD_THRESHOLD:
            head_direction = "Looking Right"
            head_display_counter = head_movement_display_frames
        elif offset < -HEAD_THRESHOLD:
            head_direction = "Looking Left"
            head_display_counter = head_movement_display_frames
        else:
            head_direction = "Facing Center"
            
            
        # LEFT EYE
        left_eye_indices = [33, 160, 158, 133, 153, 144]
        left_eye_points = []

        for idx in left_eye_indices:
            x = int(landmarks[idx].x * w)
            y = int(landmarks[idx].y * h)
            left_eye_points.append((x, y))
            cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

        # RIGHT EYE
        right_eye_indices = [362, 385, 387, 263, 373, 380]
        right_eye_points = []

        for idx in right_eye_indices:
            x = int(landmarks[idx].x * w)
            y = int(landmarks[idx].y * h)
            right_eye_points.append((x, y))
            cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

        # Compute EAR
        ear_left = calculate_ear(left_eye_points)
        ear_right = calculate_ear(right_eye_points)
        ear = (ear_left + ear_right) / 2
        
        # blink logic
        if ear < EAR_THRESHOLD:
            blink_counter += 1
        else:
            if blink_counter >= BLINK_FRAMES:
                blink_display_counter = blink_display_frames
            blink_counter = 0
            
        if head_display_counter > 0:
            if head_direction == "Facing Center":
                color = (0, 255, 0)
            else:
                color = (0, 0, 255)
            
            cv2.putText(frame, head_direction, (20, 120),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            color, 2)
            head_display_counter -= 1
        
        # display EAR value
        cv2.putText(frame, f"EAR: {ear:.2f}", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1,
            (255, 0, 0), 2)   
    
        if blink_display_counter > 0:
            cv2.putText(frame, "Blink Detected", (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1,
                        (0, 255, 0), 2)
            blink_display_counter -= 1
        
    else:
        blink_counter = 0
    
    # show frame bit + exit
    cv2.imshow("Liveness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()

####    `mp_image = mp.Image(...)`

Wraps numpy image into MediaPipe format.

#### `timestamp = int(time.time() * 1000)`

Required for `VIDEO` mode. <br>
MediaPipe tracks time between frames.

#### `result = landmarker.detect_for_video(mp_image, timestamp)`

Returns:

- result.face_landmarks

- Each face has 468 landmarks

#### `if result.face_landmarks:`

Check if face detected.

#### `nose = landmarks[1]`

Landmark index 1 = nose tip.

#### `nose_x = int(nose.x * w)`

To convert normalized coordinates

Why multiply?

- MediaPipe gives values from 0 → 1 (relative position).
- Multiply by width/height to convert to pixel space.

#### offset = nose_x - frame_center_x

| Condition           | Meaning       |
| ------------------- | ------------- |
| offset > threshold  | Looking right |
| offset < -threshold | Looking left  |
| else                | Center        |

This works because nose shifts left/right when head rotates.

#### Left Eye

`left_eye_indices = [33, 160, 158, 133, 153, 144]`

#### Right Eye

`right_eye_indices = [362, 385, 387, 263, 373, 380]`

These are MediaPipe’s fixed eye landmark indices.

#### Each point is extracted:

`x = int(landmarks[idx].x * w)` <br>
`y = int(landmarks[idx].y * h)`

Why multiply?

Because landmarks are normalized (0–1).

#### EAR Computation

    ear_left = calculate_ear(left_eye_points)
    ear_right = calculate_ear(right_eye_points)
    ear = (ear_left + ear_right) / 2

Using both eyes makes detection more stable.

#### Blink Logic

    if ear < EAR_THRESHOLD:
        blink_counter += 1

If closed → count frames.

When eye opens:

    if blink_counter >= BLINK_FRAMES:
        blink_display_counter = blink_display_frames

This prevents false blinks from noise.

#### Display Logic

Show Head Direction

`cv2.putText(...)`

- Green → Center
- Red → Left/Right

Show EAR

`cv2.putText(frame, f"EAR: {ear:.2f}", ...)`

Displays live EAR value.

Show Blink Message <br>
if blink_display_counter > 0:

Displays:

Blink Detected

#### If No Face
else:
    blink_counter = 0

Resets blink detection.

### Extracting Eye Lankmarks

Each face has:

- 468 landmarks

Each landmark contains:

- x (normalized 0–1)

- y (normalized 0–1)

- z (depth)

Left eye landmarks (common indices used):

`[33, 160, 158, 133, 153, 144]`

Right eye landmarks:

`[362, 385, 387, 263, 373, 380]`


#### `for idx in right_eye_indices:`

Convert normalized landmarks to pixel